# Nested tuples and the category $\mathbf{Nest}$

A companion to `tract.nested_tuple` and `tract.nest_morphism`, part of the code
accompanying *Categorical Foundations for CuTe Layouts* (Colfax Research).

This notebook covers:

1. **Nested tuples** — the objects of the categories $\mathbf{Nest}$ and $\mathbf{Ref}$ —
   and their basic invariants (flattening, length, rank, depth, size);
2. **profiles and refinement** of nested tuples;
3. the category $\mathbf{Nest}$ of nested tuple morphisms and how a morphism encodes a
   (nested) CuTe layout, cross-validated against NVIDIA's pure-Python CuTe reference
   implementation **pycute**;
4. the **flattening functor** $\mathbf{Nest} \to \mathbf{Tuple}$;
5. nested **coalescence**, **logical divide**, and **logical product**;
6. **pullback** and **pushforward** of a morphism along a refinement;
7. **mutual refinement** and **weak composition**;
8. TikZ export of morphism diagrams.

In [1]:
from tract import (
    NestedTuple,
    NestMorphism,
    TupleMorphism,
    mutual_refinement,
    weak_composite,
)
from tract.backends import pycute as pk

## 1. Nested tuples

A `NestedTuple` wraps a positive integer or an arbitrarily nested parenthesization of
positive integers, e.g. $S = ((4,(2,2)),\,8,\,(3,3))$. These are the objects of both
$\mathbf{Nest}$ and $\mathbf{Ref}$. The basic invariants:

- **flattening** `flatten()` — the flat tuple of all integer entries, read left to right;
- **length** — the number of entries in the flattening;
- **rank** — the number of top-level modes;
- **depth** — an integer has depth $0$, a flat tuple depth $1$, and a nested tuple
  $1 + $ the maximum depth of its modes;
- **size** — the product of all entries.

In [2]:
S = NestedTuple(((4, (2, 2)), 8, (3, 3)))
print("S          =", S)
print("flatten(S) =", S.flatten())
print("length(S)  =", S.length())
print("rank(S)    =", S.rank())
print("depth(S)   =", S.depth())
print("size(S)    =", S.size())

S          = ((4,(2,2)),8,(3,3))
flatten(S) = (4, 2, 2, 8, 3, 3)
length(S)  = 6
rank(S)    = 3
depth(S)   = 3
size(S)    = 1152


Top-level modes are extracted with `mode(i)` and individual entries of the flattening
with `entry(i)`; both are 1-indexed, matching the paper's conventions.

In [3]:
for i in range(1, S.rank() + 1):
    print(f"mode {i} of S:", S.mode(i), "  (depth", str(S.mode(i).depth()) + ")")
print("entry 3 of S:", S.entry(3))

mode 1 of S: (4,(2,2))   (depth 2)
mode 2 of S: 8   (depth 0)
mode 3 of S: (3,3)   (depth 1)
entry 3 of S: 2


## 2. Profiles, congruence, and substitution

The **profile** of a nested tuple is its bare parenthesization, obtained by
substituting $0$ for every entry. Two nested tuples are **congruent** when they have
the same profile. `sub(values)` substitutes a flat tuple of values (of the same length)
back into the parenthesization; it is the basic tool for transporting data along a
fixed profile.

In [4]:
print("profile of S:", S.profile())

T = NestedTuple(((1, (2, 3)), 4, (5, 6)))
print("S congruent to T:", S.is_congruent_to(T))
print("S congruent to (1,2,3):", S.is_congruent_to(NestedTuple((1, 2, 3))))

print("substitution:", S.sub((10, 20, 30, 40, 50, 60)))

profile of S: ((0,(0,0)),0,(0,0))
S congruent to T: True
S congruent to (1,2,3): False
substitution: ((10,(20,30)),40,(50,60))


## 3. Refinement

A nested tuple $S$ **refines** $T$, written $S \twoheadrightarrow T$, when $S$ is
obtained from $T$ by replacing entries of $T$ with nested factorizations of them.
Precisely, `S.refines(T)` holds when

1. $S = T$, or
2. $T = \mathrm{size}(S)$ (a single integer), or
3. $\mathrm{rank}(S) = \mathrm{rank}(T)$ and mode $i$ of $S$ refines mode $i$ of $T$
   for all $i$.

In [5]:
T      = NestedTuple((6, 6))
Tprime = NestedTuple(((2, 3), (2, 3)))
print(f"{Tprime} refines {T}:", Tprime.refines(T))
print(f"{Tprime} refines 36:", Tprime.refines(NestedTuple(36)))
print(f"{T} refines {Tprime}:", T.refines(Tprime))
print(f"{T} is refined by {Tprime}:", T.is_refined_by(Tprime))

((2,3),(2,3)) refines (6,6): True
((2,3),(2,3)) refines 36: True
(6,6) refines ((2,3),(2,3)): False
(6,6) is refined by ((2,3),(2,3)): True


When $S$ refines $T$, the **$i$-th relative mode** of $S$ with respect to $T$ is the
subtree of $S$ that refines the $i$-th entry of $T$. The relative modes reassemble into
the **relative flattening**, a nested tuple congruent in rank to the flattening of $T$,
and the refinement determines an **underlying map** sending each entry of $S$ to the
entry of $T$ it refines.

In [6]:
U      = NestedTuple((12, 5, 6))
Uprime = NestedTuple(((2, (3, 2)), 5, (3, 2)))
print(f"{Uprime} refines {U}:", Uprime.refines(U))
for i in range(1, U.length() + 1):
    print(f"relative mode {i}:", Uprime.relative_mode(i, U))
print("relative flattening:", Uprime.relative_flattening(U))
print("underlying map:     ", Uprime.underlying_map(U))

((2,(3,2)),5,(3,2)) refines (12,5,6): True
relative mode 1: (2,(3,2))
relative mode 2: 5
relative mode 3: (3,2)
relative flattening: ((2,(3,2)),5,(3,2))
underlying map:      (1, 1, 1, 2, 3, 3)


## 4. The category $\mathbf{Nest}$

Objects of $\mathbf{Nest}$ are nested tuples. A morphism $f\colon S \to T$ is a
`NestMorphism(domain, codomain, map)`: it lies over a morphism
$\alpha\colon \langle m\rangle_* \to \langle n\rangle_*$ of based finite sets, where
$m$ and $n$ are the lengths of $S$ and $T$. The tuple `map` records $\alpha$ on the
flattenings: entry $i$ of the domain is sent to entry `map[i-1]` of the codomain,
with $0$ denoting the basepoint $*$. The defining condition is
$s_i = t_{\alpha(i)}$ whenever $\alpha(i) \neq *$: matched entries must be equal.

In [7]:
f = NestMorphism(
    domain=((4, 2), (3, 7)),
    codomain=(3, (4, 5), 2),
    map=(2, 4, 1, 0),
)
print(f)
print("domain:  ", f.domain, "  codomain:", f.codomain)
print("size:    ", f.size(), "  cosize: ", f.cosize())

((4,2),(3,7)) --(2, 4, 1, 0)--> (3,(4,5),2)
domain:   ((4,2),(3,7))   codomain: (3,(4,5),2)
size:     168   cosize:  120


In [8]:
# The condition s_i = t_alpha(i) is validated by the constructor.
try:
    NestMorphism(domain=(4, 2), codomain=(3, 5), map=(1, 2))
except ValueError as e:
    print("ValueError:", e)

ValueError: Must satisfy s_i = t_α(i) for all i


Identity morphisms and composition make $\mathbf{Nest}$ a category. As everywhere in
`tract`, `f.compose(g)` is the diagrammatic composite $g \circ f\colon S \to T \to V$.

In [9]:
g = NestMorphism(
    domain=(3, (4, 5), 2),
    codomain=((3, 2), (5, 4)),
    map=(1, 4, 3, 2),
)
gf = f.compose(g)
print("g o f =", gf)

id_T = NestMorphism.identity(f.codomain)
print("identity on", f.codomain, ":", id_T)
print("unit laws hold:",
      f.compose(id_T) == f
      and NestMorphism.identity(f.domain).compose(f) == f)

g o f = ((4,2),(3,7)) --(4, 2, 1, 0)--> ((3,2),(5,4))
identity on (3,(4,5),2) : (3,(4,5),2) --(1, 2, 3, 4)--> (3,(4,5),2)
unit laws hold: True


## 5. Nested tuple morphisms encode CuTe layouts

A morphism $f\colon S \to T$ encodes the CuTe layout $L_f$ with shape $S$ and, for each
domain entry, the stride given by the product of the codomain entries strictly before
its image (and stride $0$ for basepoint entries). `tract.backends.pycute` computes
$L_f$ as a genuine `pycute.Layout`, using NVIDIA's official pure-Python CuTe
reference implementation.

In [10]:
L_f = pk.compute_layout(f)
print("f   =", f)
print("L_f =", L_f)

f   = ((4,2),(3,7)) --(2, 4, 1, 0)--> (3,(4,5),2)
L_f = ((4, 2), (3, 7)):((3, 60), (1, 0))


Conversely, every **tractable** layout $L$ has a standard representation: a nested
tuple morphism $f_L$ with $L_{f_L} = L$. `compute_Nest_morphism` computes it, and the
round trip recovers the layout.

In [11]:
L = pk.Layout(((4, 2), (3, 7)), ((3, 60), (1, 0)))
print("L =", L, "  tractable:", pk.is_tractable(L))

f_L = pk.compute_Nest_morphism(L)
print("f_L      =", f_L)
print("L_{f_L}  =", pk.compute_layout(f_L))
print("round trip recovers L:", pk.layouts_agree(pk.compute_layout(f_L), L))

L = ((4, 2), (3, 7)):((3, 60), (1, 0))   tractable: True
f_L      = ((4,2),(3,7)) --(2, 4, 1, 0)--> (3,4,5,2)
L_{f_L}  = ((4, 2), (3, 7)):((3, 60), (1, 0))
round trip recovers L: True


## 6. The flattening functor $\mathbf{Nest} \to \mathbf{Tuple}$

Flattening domain and codomain (keeping the same underlying map) sends a nested tuple
morphism to a `TupleMorphism`, the flat notion studied first in the paper. This
assignment is a functor: it preserves identities and composition. On layouts it
corresponds to flattening the shape. `flatten_codomain()` flattens only the codomain,
which leaves the encoded layout unchanged.

In [12]:
flat_f = f.flatten()
print("f.flatten() =", flat_f)
print("type:", type(flat_f).__name__)
print("f.flatten_codomain() =", f.flatten_codomain())

f.flatten() = (4, 2, 3, 7) --(2, 4, 1, 0)--> (3, 4, 5, 2)
type: TupleMorphism
f.flatten_codomain() = ((4,2),(3,7)) --(2, 4, 1, 0)--> (3,4,5,2)


In [13]:
# Functoriality: flattening commutes with composition.
print("(g o f).flatten() == g.flatten() o f.flatten():",
      f.compose(g).flatten() == f.flatten().compose(g.flatten()))
print("id_T.flatten() is the identity:", id_T.flatten().is_identity())

(g o f).flatten() == g.flatten() o f.flatten(): True
id_T.flatten() is the identity: True


## 7. Nested coalescence

**Coalescence** merges adjacent modes of a morphism whenever the encoded layout
function does not distinguish them, producing the shortest morphism encoding the same
layout up to congruence. On the layout side this is CuTe's `coalesce`. We cross-check
the categorical operation against pycute, following the recipe of
`tests/test_cross_validation.py`: coalescing the morphism and then taking its layout
agrees with taking the layout and coalescing it.

In [14]:
h = NestMorphism(
    domain=((2, 4), (8, 3)),
    codomain=(2, 4, 8, 3, 5),
    map=(1, 2, 3, 4),
)
print("h            =", h)
print("h.coalesce() =", h.coalesce())

h            = ((2,4),(8,3)) --(1, 2, 3, 4)--> (2,4,8,3,5)
h.coalesce() = 192 --(1,)--> (192,5)


In [15]:
layout_h = pk.compute_layout(h)
coalesce_then_layout = pk.compute_layout(h.coalesce())
layout_then_coalesce = pk.coalesce_layout(layout_h)
print("L_h                  =", layout_h)
print("L_{coalesce(h)}      =", coalesce_then_layout)
print("coalesce(L_h)        =", layout_then_coalesce)
print("agree:", pk.layouts_agree(coalesce_then_layout, layout_then_coalesce))

L_h                  = ((2, 4), (8, 3)):((1, 2), (8, 64))
L_{coalesce(h)}      = 192:1
coalesce(L_h)        = 192:1
agree: True


## 8. Complements and logical divide

A morphism $g\colon X \to S$ with no basepoint entries is **complementable**: its
complement $g^c$ collects the codomain entries missed by $g$, and the concatenation
$(g, g^c)$ is an isomorphism onto $S$. The **logical divide** of $f\colon S \to T$ by a
tiler $g\colon X \to S$ is

$$ f \,/\, g \;=\; f \circ (g, g^c)\colon (X, X^c) \to T, $$

which reorganizes $f$'s domain into the tile $X$ and the remainder $X^c$ — the
categorical counterpart of CuTe's `logical_divide`.

In [16]:
f2 = NestMorphism(
    domain=((2, 2), (3, 2)),
    codomain=(2, 2, 3, 2, 5),
    map=(1, 2, 3, 4),
)
tiler = NestMorphism(domain=(2, 3), codomain=((2, 2), (3, 2)), map=(1, 3))
print("tiler complementable:", tiler.is_complementable())
print("tiler complement:    ", tiler.complement())
print("complementary pair:  ", tiler.is_complementary_to(tiler.complement()))

quotient = f2.logical_divide(tiler)
print("f2 / tiler =", quotient)

tiler complementable: True
tiler complement:     (2,2) --(2, 4)--> ((2,2),(3,2))
complementary pair:   True
f2 / tiler = ((2,3),(2,2)) --(1, 3, 2, 4)--> (2,2,3,2,5)


In [17]:
# Cross-check against the CuTe layout algebra: compose L_{f2} with the
# concatenation of L_tiler and its layout-level complement.
layout_f2 = pk.compute_layout(f2)
layout_tiler = pk.compute_layout(tiler)
tiler_complement = pk.flat_complement(pk.flatten_layout(layout_tiler), f2.size())
quotient_layout = pk.compose_layouts(
    layout_f2, pk.concatenate(layout_tiler, tiler_complement)
)
print("L_{f2 / tiler}          =", pk.compute_layout(quotient))
print("logical divide of L_f2  =", quotient_layout)
print("agree up to coalescence:",
      pk.layouts_agree(pk.coalesce_layout(pk.compute_layout(quotient)),
                       pk.coalesce_layout(quotient_layout)))

L_{f2 / tiler}          = ((2, 3), (2, 2)):((1, 4), (2, 12))
logical divide of L_f2  = ((2, 3), (1, 2, 2)):((1, 4), (1, 2, 12))
agree up to coalescence: True


## 9. Logical product

Dually, the **logical product** of a complementable $f\colon S \to T$ with
$g\colon X \to S^c$ (where $S^c$ is the domain of $f$'s complement) is

$$ f * g \;=\; (f,\; f^c \circ g)\colon (S, X) \to T, $$

which repeats the tile $f$ according to the pattern $g$ — the categorical counterpart
of CuTe's `logical_product`. The cross-check compares $L_{f * g}$ with pycute's
`logical_product` of the two layouts.

In [18]:
tile = NestMorphism(domain=(2, 3), codomain=(2, 3, 4, 5), map=(1, 2))
pattern = NestMorphism(domain=(4,), codomain=(4, 5), map=(1,))
product = tile.logical_product(pattern)
print("tile complement:", tile.complement())
print("tile * pattern =", product)

tile complement: (4,5) --(3, 4)--> (2,3,4,5)
tile * pattern = ((2,3),(4)) --(1, 2, 3)--> (2,3,4,5)


In [19]:
layout_tile = pk.compute_layout(tile)
layout_pattern = pk.compute_layout(pattern)
product_layout = pk.logical_product_layouts(layout_tile, layout_pattern)
print("L_{tile * pattern}              =", pk.compute_layout(product))
print("logical_product(L_tile, L_pat)  =", product_layout)
print("agree:", pk.layouts_agree(pk.compute_layout(product), product_layout))

L_{tile * pattern}              = ((2, 3), (4,)):((1, 2), (6,))
logical_product(L_tile, L_pat)  = ((2, 3), (4,)):((1, 2), (6,))
agree: True


## 10. Pullback along a refinement of the codomain

Given $f\colon S \to T$ and a refinement $T' \twoheadrightarrow T$, the **pullback**
$f' = f^*(T')\colon S' \to T'$ replaces each domain entry hit by $f$ with the relative
mode of $T'$ refining its image (basepoint entries pass through untouched), and remaps
entries accordingly. The domain $S'$ refines $S$, completing a square

$$\begin{array}{ccc}
S' & \xrightarrow{\;f'\;} & T' \\
\downarrow & & \downarrow \\
S & \xrightarrow{\;f\;} & T
\end{array}$$

In [20]:
p = NestMorphism(domain=(6, 4), codomain=(4, 6), map=(2, 1))
Tprime = NestedTuple(((2, 2), (2, 3)))
p_pulled = p.pullback_along(Tprime)
print("p              =", p)
print("refinement T'  =", Tprime, " refines", p.codomain, ":", Tprime.refines(p.codomain))
print("pullback of p  =", p_pulled)
print("new domain refines old:", p_pulled.domain.refines(p.domain))

p              = (6,4) --(2, 1)--> (4,6)
refinement T'  = ((2,2),(2,3))  refines (4,6) : True
pullback of p  = ((2,3),(2,2)) --(3, 4, 1, 2)--> ((2,2),(2,3))
new domain refines old: True


## 11. Pushforward along a refinement of the domain

Dually, given $g\colon U \to V$ and a refinement $U' \twoheadrightarrow U$, the
**pushforward** $g' = g_*(U')\colon U' \to V'$ refines each codomain entry in the image
of $g$ by the relative mode of $U'$ over its preimage, leaving the rest of $V$
unchanged; $V'$ refines $V$.

In [21]:
q = NestMorphism(domain=(6, 4), codomain=(4, 6, 2), map=(2, 1))
Uprime = NestedTuple(((3, 2), 4))
q_pushed = q.pushforward_along(Uprime)
print("q               =", q)
print("refinement U'   =", Uprime, " refines", q.domain, ":", Uprime.refines(q.domain))
print("pushforward of q=", q_pushed)
print("new codomain refines old:", q_pushed.codomain.refines(q.codomain))

q               = (6,4) --(2, 1)--> (4,6,2)
refinement U'   = ((3,2),4)  refines (6,4) : True
pushforward of q= ((3,2),4) --(2, 3, 1)--> (4,(3,2),2)
new codomain refines old: True


## 12. Mutual refinement

Composition of layouts $B \circ A$ makes sense even when the shape of $B$ does not
match the codomain of $A$'s morphism on the nose — CuTe only requires a divisibility
condition. Categorically this is captured by **mutual refinement**: given nested tuples
$T$ and $U$, `mutual_refinement(T, U)` produces $T'$ refining $T$ and $U'$ refining $U$
with $\mathrm{flat}(T')$ a prefix-compatible factorization of $\mathrm{flat}(U')$
(so that $T'$ includes into $U'$ entrywise). The running example from the paper:

$$T = (6,6),\quad U = (2,6,6) \;\longmapsto\; T' = ((2,3),(2,3)),\quad U' = (2,(3,2),(3,2)).$$

In [22]:
T = NestedTuple((6, 6))
U = NestedTuple((2, 6, 6))
Tprime, Uprime = mutual_refinement(T, U)
print("T' =", Tprime, "  refines T:", Tprime.refines(T))
print("U' =", Uprime, "  refines U:", Uprime.refines(U))
print("flat(T') =", Tprime.flatten())
print("flat(U') =", Uprime.flatten())

T' = ((2,3),(2,3))   refines T: True
U' = (2,(3,2),(3,2))   refines U: True
flat(T') = (2, 3, 2, 3)
flat(U') = (2, 3, 2, 3, 2)


## 13. Weak composition

Given $f\colon S \to T$ and $g\colon U \to V$ with $T$ and $U$ mutually refinable, the
**weak composite** is

$$ g \diamond f \;=\; g_*(U') \circ \iota \circ f^*(T'), $$

where $f^*(T')$ is the pullback of $f$, $g_*(U')$ the pushforward of $g$, and
$\iota\colon T' \to U'$ the entrywise inclusion of the mutual refinement. This is the
categorical model of general CuTe layout composition: coalescing $L_{g \diamond f}$
back to the profile of $S$ recovers the CuTe composition $L_g \circ L_f$.

In [23]:
f_w = NestMorphism(domain=(6, 6), codomain=(6, 6), map=(2, 1))
g_w = NestMorphism(domain=(2, 6, 6), codomain=(6, 2, 6), map=(2, 1, 3))
wc = weak_composite(f_w, g_w)
print("f            =", f_w)
print("g            =", g_w)
print("g <> f       =", wc)

f            = (6,6) --(2, 1)--> (6,6)
g            = (2,6,6) --(2, 1, 3)--> (6,2,6)
g <> f       = ((2,3),(2,3)) --(2, 4, 3, 1)--> ((3,2),2,(3,2))


In [24]:
# Cross-check against pycute: coalesce L_{g<>f} to the profile of S, and compare
# with the direct layout composition L_g o L_f.
S_profile = f_w.domain.data
weak_layout = pk.coalesce_layout(pk.compute_layout(wc), profile=S_profile)
direct_layout = pk.compose_layouts(pk.compute_layout(g_w), pk.compute_layout(f_w))
print("coalesced weak composite layout =", weak_layout)
print("L_g o L_f via pycute            =", direct_layout)
print("agree:", pk.layouts_agree(weak_layout, direct_layout))

coalesced weak composite layout = ((2, 3), (2, 3)):((3, 12), (6, 1))
L_g o L_f via pycute            = ((2, 3), (2, 3)):((3, 12), (6, 1))
agree: True


## 14. TikZ export

`NestMorphism.to_tikz()` renders the morphism as a TikZ diagram: the domain and
codomain parenthesization trees with the underlying map drawn between their leaves.
With `full_doc=True` it produces a standalone LaTeX document. Rendered examples live in
`tract/notebooks/images/` (e.g. `nest_morphism_to_tikz_example.png` and
`mutual_refinement_tikz_example.png`).

In [25]:
tikz = f.to_tikz()
print("\n".join(tikz.splitlines()[:16]))
print("...")
print(f"({len(tikz.splitlines())} lines total)")


\begin{tikzpicture}[
    entry/.style={minimum width=5mm, minimum height=7mm, inner sep = 2pt},
    maparrow/.style={|->}
]
\def\colspacing{3}
\def\rowspacing{0.8}

\node[entry] (s1) at (2.20, 0.00) {4};
\node[entry] (s2) at (2.20, 0.80) {2};
\node[entry] (s3) at (2.20, 1.60) {3};
\node[entry] (s4) at (2.20, 2.40) {7};

\node[entry] (m1) at (0.00, 0.00) {8};
\node[entry] (m2) at (0.00, 0.80) {21};

...
(37 lines total)


## Summary

- Nested tuples carry the invariants flatten / length / rank / depth / size, and a
  refinement order with relative modes.
- A `NestMorphism(domain, codomain, map)` over based finite sets, with
  $s_i = t_{\alpha(i)}$, encodes exactly a (nested, tractable) CuTe layout; the
  correspondence is implemented by `compute_layout` and `compute_Nest_morphism`.
- Flattening is a functor $\mathbf{Nest} \to \mathbf{Tuple}$.
- Coalescence, logical divide, and logical product on morphisms match CuTe's layout
  operations, cross-validated here against pycute.
- Pullback and pushforward along refinements, mutual refinement, and weak composition
  model general layout composition; the next notebook
  (`04_fact_ref_and_spans.ipynb`) repackages refinements as the categories
  $\mathbf{Fact}$ and $\mathbf{Ref}$ and organizes these squares into span and cospan
  categories.